# Sepsis Prediction — Exploratory Data Analysis
PhysioNet Challenge 2019

This notebook explores the dataset before running the full pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

plt.rcParams['figure.dpi'] = 120
DATA_DIR = '../training_setA'  # change if needed

## 1. Load a sample of patients

In [ ]:
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.psv')))[:500]  # sample 500
dfs = []
for f in files:
    pid = os.path.splitext(os.path.basename(f))[0]
    df = pd.read_csv(f, sep='|')
    df['patient_id'] = pid
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
print(f'Rows: {len(data):,}')
print(f'Patients: {data.patient_id.nunique():,}')
print(f'Septic patients: {data.groupby("patient_id")["SepsisLabel"].max().sum():,}')
data.head()

## 2. Missingness Analysis

In [ ]:
vital_cols = ['HR','O2Sat','Temp','SBP','MAP','DBP','Resp','EtCO2']
lab_cols = ['Creatinine','Lactate','WBC','Bilirubin_total','Platelets','pH','BUN']

missing_pct = data[vital_cols + lab_cols].isnull().mean() * 100

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2563EB']*len(vital_cols) + ['#DC2626']*len(lab_cols)
missing_pct.plot(kind='bar', ax=ax, color=colors)
ax.set_title('Missing Value Percentage by Feature', fontweight='bold')
ax.set_ylabel('% Missing')
ax.axhline(60, color='gray', linestyle='--', alpha=0.5, label='60% line')
ax.legend(['60% threshold', 'Vital Signs (blue)', 'Lab Values (red)'])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print(missing_pct.sort_values(ascending=False))

## 3. Sepsis Label Distribution

In [ ]:
label_counts = data['SepsisLabel'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['No Sepsis', 'Sepsis'], label_counts.values,
            color=['#2563EB', '#DC2626'])
axes[0].set_title('Label Distribution (Row Level)', fontweight='bold')
axes[0].set_ylabel('Count')

patient_labels = data.groupby('patient_id')['SepsisLabel'].max()
patient_counts = patient_labels.value_counts()
axes[1].bar(['Non-Septic', 'Septic'], patient_counts.values,
            color=['#2563EB', '#DC2626'])
axes[1].set_title('Patient-Level Sepsis Rate', fontweight='bold')
axes[1].set_ylabel('Patients')

plt.tight_layout()
plt.show()
print(f'Sepsis prevalence: {patient_counts[1]/len(patient_labels)*100:.1f}%')

## 4. ICU Stay Length Distribution

In [ ]:
stay_lengths = data.groupby('patient_id').size()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(stay_lengths, bins=50, color='#2563EB', alpha=0.8, edgecolor='white')
ax.set_xlabel('ICU Stay Length (hours)')
ax.set_ylabel('Number of Patients')
ax.set_title('Distribution of ICU Stay Lengths', fontweight='bold')
ax.axvline(stay_lengths.mean(), color='red', linestyle='--',
           label=f'Mean: {stay_lengths.mean():.1f}h')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Vital Sign Trends: Septic vs Non-Septic

In [ ]:
septic_ids    = data.groupby('patient_id')['SepsisLabel'].max()
septic_ids    = septic_ids[septic_ids == 1].index
nonseptic_ids = septic_ids[septic_ids == 0].index if False else \
                data.groupby('patient_id')['SepsisLabel'].max()
nonseptic_ids = nonseptic_ids[nonseptic_ids == 0].index

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
vitals = ['HR', 'MAP', 'Temp', 'Resp', 'O2Sat', 'SBP']

for ax, col in zip(axes.flat, vitals):
    sep_vals    = data[data.patient_id.isin(septic_ids)][col].dropna()
    nonsep_vals = data[data.patient_id.isin(nonseptic_ids)][col].dropna()
    ax.hist(nonsep_vals, bins=40, alpha=0.6, color='#2563EB',
            density=True, label='Non-Septic')
    ax.hist(sep_vals, bins=40, alpha=0.6, color='#DC2626',
            density=True, label='Septic')
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Vital Sign Distributions: Septic vs Non-Septic Patients',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()